# NAidioma — Servidor Ollama (GPU T4 x2) + Fábrica de Conteúdo dos 8 Idiomas

Use este notebook no **Kaggle (Accelerator: GPU T4 x2, Internet: ON)** para:
1. Subir o **Ollama** com o modelo open-weight da OpenAI (`gpt-oss:20b`) ou `qwen2.5:14b`.
2. Expor um túnel HTTPS seguro (`trycloudflare.com`) para colar direto no **Painel Admin do NAidioma**.
3. Gerar lições extras em lote (JSON compatível com `@naidioma/shared`) para os 8 idiomas a custo zero.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print('Ollama + Cloudflared prontos!')

In [ ]:
import os, subprocess, time, re

MODEL = os.environ.get('NAIDIOMA_MODEL', 'gpt-oss:20b')
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
env['OLLAMA_ORIGINS'] = '*'

ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print(f'Baixando modelo {MODEL} nas GPUs T4 x2...')
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('Modelo carregado com sucesso!')

In [ ]:
import subprocess, re, time

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(80):
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print('====================================================')
print('COLE ESTA URL NO PAINEL ADMIN DO NAIDIOMA:')
print(public_url)
print('====================================================')

In [ ]:
# Opcional: gerar novas lições em lote para os 8 idiomas em JSON pronto para importar
import json, urllib.request

def generate_unit_lesson(lang_code, lang_name, topic_pt, cefr='A2'):
    prompt = f'''Gere uma lição JSON para o app NAidioma (ensino de {lang_name} para brasileiros).
Tópico: {topic_pt}. Nível CEFR: {cefr}.
Inclua 4 exercícios variados (context_choice, listen_build, speak_pronounce, trap_buster) com dicas práticas em português brasileiro.
Responda APENAS JSON válido.'''
    req = urllib.request.Request(
        'http://127.0.0.1:11434/api/chat',
        data=json.dumps({
            'model': MODEL,
            'stream': False,
            'format': 'json',
            'messages': [{'role': 'user', 'content': prompt}],
        }).encode(),
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode())
        return json.loads(data['message']['content'])

sample = generate_unit_lesson('en', 'Inglês para Brasileiros', 'Negociando salário em entrevista remota', 'B2')
with open('/kaggle/working/naidioma_extra_lesson.json', 'w', encoding='utf-8') as f:
    json.dump(sample, f, ensure_ascii=False, indent=2)
print('Lição salva em /kaggle/working/naidioma_extra_lesson.json')